In [1]:
# ============================================================
# CELL 1 — Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import os
import json
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

In [8]:
# ============================================================
# CELL 3 — Paths
# ============================================================
#RAW_DIR = '/content/drive/MyDrive/UChicago/Masters/Spring/ADSP 31018/SignBridge/data/raw/signbridge_all250'
#PROCESSED_DIR = '/content/drive/MyDrive/UChicago/Masters/Spring/ADSP 31018/SignBridge/data/processed'
RAW_DIR = '/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250'
PROCESSED_DIR = '/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250'
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [9]:
# ============================================================
# CELL 4 — Reproducibility
# ============================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [10]:
# ============================================================
# CELL 5 — Device / GPU Check
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Device:", device)

if device.type == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
else:
    print("No GPU found. Use Runtime > Change runtime type > GPU.")

Device: cuda
GPU: Tesla T4
VRAM: 15.637086208 GB


In [11]:
# ============================================================
# CELL 6 — Helper Function for File Loading
# ============================================================
def load_from_processed_or_raw(fname):
    processed_path = os.path.join(PROCESSED_DIR, fname)
    raw_path = os.path.join(RAW_DIR, fname)

    if os.path.exists(processed_path):
        return processed_path
    elif os.path.exists(raw_path):
        return raw_path
    else:
        raise FileNotFoundError(f"Could not find {fname} in processed or raw folders.")

In [12]:
# ============================================================
# CELL 7 — Load Normalized Feature Arrays
# ============================================================
X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train_norm.npz'))['data']
X_val   = np.load(os.path.join(PROCESSED_DIR, 'X_val_norm.npz'))['data']
X_test  = np.load(os.path.join(PROCESSED_DIR, 'X_test_norm.npz'))['data']

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

X_train: (55642, 96, 708)
X_val:   (9291, 96, 708)
X_test:  (11826, 96, 708)


In [13]:
# ============================================================
# CELL 8 — Load Labels and Lengths
# ============================================================
y_train = np.load(load_from_processed_or_raw('y_train.npy'))
y_val   = np.load(load_from_processed_or_raw('y_val.npy'))
y_test  = np.load(load_from_processed_or_raw('y_test.npy'))

lengths_train = np.load(load_from_processed_or_raw('lengths_train.npy'))
lengths_val   = np.load(load_from_processed_or_raw('lengths_val.npy'))
lengths_test  = np.load(load_from_processed_or_raw('lengths_test.npy'))

print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

print("lengths_train:", lengths_train.shape)
print("lengths_val:  ", lengths_val.shape)
print("lengths_test: ", lengths_test.shape)

y_train: (55642,)
y_val:   (9291,)
y_test:  (11826,)
lengths_train: (55642,)
lengths_val:   (9291,)
lengths_test:  (11826,)


In [14]:
# ============================================================
# CELL 9 — Load Metadata and Label Mappings
# ============================================================
with open(load_from_processed_or_raw('metadata.json')) as f:
    metadata = json.load(f)

if 'label_to_word' in metadata:
    idx_to_sign = {int(k): v for k, v in metadata['label_to_word'].items()}
    sign_to_idx = {v: k for k, v in idx_to_sign.items()}

elif 'labels' in metadata:
    idx_to_sign = {int(k): v for k, v in metadata['labels'].items()}
    sign_to_idx = {v: k for k, v in idx_to_sign.items()}

elif 'signs' in metadata:
    idx_to_sign = {int(k): v for k, v in enumerate(metadata['signs'])}
    sign_to_idx = {v: k for k, v in idx_to_sign.items()}

else:
    raise KeyError("Could not find label mappings in metadata.")

print("Total classes:", len(idx_to_sign))
print("Sample labels:", list(idx_to_sign.items())[:10])

Total classes: 250
Sample labels: [(0, 'TV'), (1, 'after'), (2, 'airplane'), (3, 'all'), (4, 'alligator'), (5, 'animal'), (6, 'another'), (7, 'any'), (8, 'apple'), (9, 'arm')]


In [15]:
# ============================================================
# CELL 10 — Randomly Select 25 Signs
# ============================================================
N_PILOT_SIGNS = 25

all_signs = list(sign_to_idx.keys())

random.seed(SEED)
PILOT_SIGNS = random.sample(all_signs, N_PILOT_SIGNS)

print(f"Selected {len(PILOT_SIGNS)} signs:")
for i, sign in enumerate(PILOT_SIGNS):
    print(f"{i:2d}: {sign}")

Selected 25 signs:
 0: outside
 1: book
 2: another
 3: sad
 4: empty
 5: drawer
 6: dirty
 7: can
 8: room
 9: blue
10: please
11: yourself
12: uncle
13: make
14: better
15: napkin
16: have
17: apple
18: any
19: bird
20: dad
21: doll
22: kitty
23: noisy
24: yucky


In [16]:
# ============================================================
# CELL 11 — Map Original Labels to New 0–24 Labels
# ============================================================
pilot_original_idx = [sign_to_idx[s] for s in PILOT_SIGNS]

orig_to_new = {
    orig: new for new, orig in enumerate(pilot_original_idx)
}

new_to_orig = {
    new: orig for orig, new in orig_to_new.items()
}

print("Original label → New label:")
for orig, new in orig_to_new.items():
    print(f"{idx_to_sign[orig]:20s} orig={orig:3d} → new={new:2d}")

Original label → New label:
outside              orig=163 → new= 0
book                 orig= 28 → new= 1
another              orig=  6 → new= 2
sad                  orig=189 → new= 3
empty                orig= 70 → new= 4
drawer               orig= 62 → new= 5
dirty                orig= 57 → new= 6
can                  orig= 35 → new= 7
room                 orig=188 → new= 8
blue                 orig= 26 → new= 9
please               orig=173 → new=10
yourself             orig=246 → new=11
uncle                orig=228 → new=12
make                 orig=139 → new=13
better               orig= 22 → new=14
napkin               orig=151 → new=15
have                 orig=108 → new=16
apple                orig=  8 → new=17
any                  orig=  7 → new=18
bird                 orig= 23 → new=19
dad                  orig= 55 → new=20
doll                 orig= 59 → new=21
kitty                orig=129 → new=22
noisy                orig=154 → new=23
yucky                orig=247 → new=

In [17]:
# ============================================================
# CELL 12 — Filter Dataset to 25 Signs
# ============================================================
def filter_split(X, y, lengths, original_indices, orig_to_new):
    mask = np.isin(y, original_indices)

    X_f = X[mask]
    y_f = np.array([orig_to_new[int(label)] for label in y[mask]], dtype=np.int64)
    l_f = lengths[mask]

    return X_f, y_f, l_f


X_tr, y_tr, l_tr = filter_split(
    X_train, y_train, lengths_train, pilot_original_idx, orig_to_new
)

X_vl, y_vl, l_vl = filter_split(
    X_val, y_val, lengths_val, pilot_original_idx, orig_to_new
)

X_te, y_te, l_te = filter_split(
    X_test, y_test, lengths_test, pilot_original_idx, orig_to_new
)

N_CLASSES = len(PILOT_SIGNS)

print("Filtered dataset:")
print("Train:", X_tr.shape, y_tr.shape)
print("Val:  ", X_vl.shape, y_vl.shape)
print("Test: ", X_te.shape, y_te.shape)
print("N_CLASSES:", N_CLASSES)

Filtered dataset:
Train: (5611, 96, 708) (5611,)
Val:   (927, 96, 708) (927,)
Test:  (1181, 96, 708) (1181,)
N_CLASSES: 25


In [18]:
# ============================================================
# CELL 13 — Save 25-Sign Label Map
# ============================================================
pilot_label_map = {
    'n_classes': N_CLASSES,
    'signs': PILOT_SIGNS,
    'new_to_sign': {str(i): s for i, s in enumerate(PILOT_SIGNS)},
    'sign_to_new': {s: i for i, s in enumerate(PILOT_SIGNS)},
    'orig_to_new': {str(k): int(v) for k, v in orig_to_new.items()},
    'new_to_orig': {str(k): int(v) for k, v in new_to_orig.items()},
    'seed': SEED,
}

label_map_path = os.path.join(PROCESSED_DIR, 'pilot25_label_map.json')

with open(label_map_path, 'w') as f:
    json.dump(pilot_label_map, f, indent=2)

print("Saved:", label_map_path)

Saved: /content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/pilot25_label_map.json


In [19]:
# ============================================================
# CELL 14 — Compute Class Weights for 25 Signs
# ============================================================
classes = np.arange(N_CLASSES)

class_weights_25 = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_tr
).astype(np.float32)

print("Class weights:")
print(class_weights_25)

np.save(os.path.join(PROCESSED_DIR, 'class_weights_25.npy'), class_weights_25)
print("Saved: class_weights_25.npy")

Class weights:
[1.0019643  1.1056157  1.1569072  0.86992246 1.24       1.1335354
 0.93128633 1.3439521  1.2002139  1.068762   0.905      0.98872244
 0.81318843 0.8906349  1.1392894  0.85992336 1.2066667  0.905
 1.226448   0.83125925 0.94302523 0.8281919  0.908664   1.1222
 0.91983604]
Saved: class_weights_25.npy


In [20]:

# ============================================================
# CELL 15 — Dataset Class
# ============================================================
class SignDataset(Dataset):
    def __init__(self, X, y, lengths, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        l = self.lengths[idx]

        if self.augment:
            x, l = self._augment(x, l)

        return x, y, l

    def _augment(self, x, l):
        x = x.clone()
        l_int = int(l.item())

        if torch.rand(1).item() < 0.5:
            rate = 0.8 + torch.rand(1).item() * 0.4
            new_len = max(1, int(round(l_int * rate)))

            real = x[:l_int]
            old_idx = torch.linspace(0, l_int - 1, new_len)

            lo = old_idx.floor().long().clamp(0, l_int - 1)
            hi = old_idx.ceil().long().clamp(0, l_int - 1)
            frac = (old_idx - lo.float()).unsqueeze(1)

            stretched = real[lo] * (1 - frac) + real[hi] * frac

            out = torch.zeros_like(x)
            keep = min(new_len, x.shape[0])
            out[:keep] = stretched[:keep]

            x = out
            l_int = keep

        if torch.rand(1).item() < 0.5:
            noise = torch.randn(l_int, x.shape[1]) * 0.001
            x[:l_int] += noise

        if torch.rand(1).item() < 0.5:
            mask = torch.rand(l_int) < 0.10
            x[:l_int][mask] = 0.0

        return x, torch.tensor(l_int, dtype=torch.long)

In [21]:
# ============================================================
# CELL 16 — Create DataLoaders
# ============================================================
BATCH = 64

train_dataset = SignDataset(X_tr, y_tr, l_tr, augment=True)
val_dataset   = SignDataset(X_vl, y_vl, l_vl, augment=False)
test_dataset  = SignDataset(X_te, y_te, l_te, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

x_batch, y_batch, l_batch = next(iter(train_loader))

print("Batch X:", x_batch.shape)
print("Batch y:", y_batch.shape)
print("Batch lengths:", l_batch.shape)
print("Label range:", y_batch.min().item(), "to", y_batch.max().item())

Batch X: torch.Size([64, 96, 708])
Batch y: torch.Size([64])
Batch lengths: torch.Size([64])
Label range: 0 to 24


In [22]:
# ============================================================
# CELL 17 — MLP Model
# ============================================================
class SignMLP(nn.Module):
    def __init__(self, input_size=708, seq_len=96, hidden=512, n_classes=25, dropout=0.3):
        super().__init__()

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size * seq_len, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, x, lengths=None):
        return self.net(x)

In [23]:
# ============================================================
# CELL 18 — CNN Model
# ============================================================
class DepthwiseSeparableConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        super().__init__()

        self.depthwise = nn.Conv1d(
            in_channels,
            in_channels,
            kernel_size=kernel_size,
            padding=padding,
            groups=in_channels,
            bias=False
        )

        self.pointwise = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class SignCNN(nn.Module):
    def __init__(self, input_size=708, channels=256, n_classes=25, dropout=0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_size, channels),
            nn.LayerNorm(channels),
        )

        self.block1 = self._make_block(channels, kernel=3, dropout=dropout)
        self.block2 = self._make_block(channels, kernel=5, dropout=dropout)
        self.block3 = self._make_block(channels, kernel=7, dropout=dropout)

        self.classifier = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, n_classes)
        )

    def _make_block(self, channels, kernel, dropout):
        padding = kernel // 2

        return nn.ModuleDict({
            'conv': DepthwiseSeparableConv1d(channels, channels, kernel, padding),
            'bn': nn.BatchNorm1d(channels),
            'act': nn.GELU(),
            'drop': nn.Dropout(dropout),
        })

    def _apply_block(self, block, x):
        residual = x
        x = block['conv'](x)
        x = block['bn'](x)
        x = block['act'](x)
        x = block['drop'](x)
        return x + residual

    def forward(self, x, lengths=None):
        x = self.input_proj(x)
        x = x.transpose(1, 2)

        x = self._apply_block(self.block1, x)
        x = self._apply_block(self.block2, x)
        x = self._apply_block(self.block3, x)

        if lengths is not None:
            mask = torch.arange(x.shape[2], device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
            mask = mask.unsqueeze(1).float()
            x = (x * mask).sum(dim=2) / mask.sum(dim=2).clamp(min=1)
        else:
            x = x.mean(dim=2)

        return self.classifier(x)

In [24]:
# ============================================================
# CELL 19 — LSTM Model
# ============================================================
class SignLSTM(nn.Module):
    def __init__(self, input_size=708, hidden=256, layers=2, n_classes=25, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.norm = nn.LayerNorm(hidden * 2)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x, lengths=None):
        out, _ = self.lstm(x)

        if lengths is not None:
            idx = (lengths - 1).clamp(min=0)
            out = out[torch.arange(len(idx), device=x.device), idx]
        else:
            out = out[:, -1]

        return self.fc(self.drop(self.norm(out)))

In [25]:
# ============================================================
# CELL 20 — GRU Model
# ============================================================
class SignGRU(nn.Module):
    def __init__(self, input_size=708, hidden=256, layers=2, n_classes=25, dropout=0.3):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.norm = nn.LayerNorm(hidden * 2)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x, lengths=None):
        out, _ = self.gru(x)

        if lengths is not None:
            idx = (lengths - 1).clamp(min=0)
            out = out[torch.arange(len(idx), device=x.device), idx]
        else:
            out = out[:, -1]

        return self.fc(self.drop(self.norm(out)))

In [26]:
# ============================================================
# CELL 21 — CNN + LSTM Model
# ============================================================
class SignCNNLSTM(nn.Module):
    def __init__(self, input_size=708, conv_channels=256, hidden=256, n_classes=25, dropout=0.3):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(input_size, conv_channels),
            nn.LayerNorm(conv_channels),
            nn.GELU()
        )

        self.conv = nn.Sequential(
            nn.Conv1d(conv_channels, conv_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(conv_channels),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Conv1d(conv_channels, conv_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(conv_channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.norm = nn.LayerNorm(hidden * 2)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x, lengths=None):
        x = self.proj(x)
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)

        out, _ = self.lstm(x)

        if lengths is not None:
            idx = (lengths - 1).clamp(min=0)
            out = out[torch.arange(len(idx), device=x.device), idx]
        else:
            out = out[:, -1]

        return self.fc(self.drop(self.norm(out)))

In [27]:
# ============================================================
# CELL 22 — CNN + GRU Model
# ============================================================
class SignCNNGRU(nn.Module):
    def __init__(self, input_size=708, conv_channels=256, hidden=256, n_classes=25, dropout=0.3):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(input_size, conv_channels),
            nn.LayerNorm(conv_channels),
            nn.GELU()
        )

        self.conv = nn.Sequential(
            nn.Conv1d(conv_channels, conv_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(conv_channels),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Conv1d(conv_channels, conv_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(conv_channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.gru = nn.GRU(
            input_size=conv_channels,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.norm = nn.LayerNorm(hidden * 2)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x, lengths=None):
        x = self.proj(x)
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)

        out, _ = self.gru(x)

        if lengths is not None:
            idx = (lengths - 1).clamp(min=0)
            out = out[torch.arange(len(idx), device=x.device), idx]
        else:
            out = out[:, -1]

        return self.fc(self.drop(self.norm(out)))

In [28]:
# ============================================================
# CELL 23 — BiLSTM + Attention Model
# ============================================================
class SignBiLSTMAttention(nn.Module):
    def __init__(self, input_size=708, hidden=256, n_classes=25, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.attn = nn.Linear(hidden * 2, 1)
        self.norm = nn.LayerNorm(hidden * 2)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x, lengths=None):
        out, _ = self.lstm(x)

        scores = self.attn(out).squeeze(-1)

        if lengths is not None:
            mask = torch.arange(out.shape[1], device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)
            scores = scores.masked_fill(mask, -1e9)

        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        pooled = (out * weights).sum(dim=1)

        return self.fc(self.drop(self.norm(pooled)))

In [29]:
# ============================================================
# CELL 24 — TCN Model
# ============================================================
class TemporalBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilation=1, dropout=0.3):
        super().__init__()

        padding = dilation * (kernel_size - 1) // 2

        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.net(x)


class SignTCN(nn.Module):
    def __init__(self, input_size=708, channels=256, n_classes=25, dropout=0.3):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(input_size, channels),
            nn.LayerNorm(channels),
            nn.GELU()
        )

        self.tcn = nn.Sequential(
            TemporalBlock(channels, dilation=1, dropout=dropout),
            TemporalBlock(channels, dilation=2, dropout=dropout),
            TemporalBlock(channels, dilation=4, dropout=dropout),
            TemporalBlock(channels, dilation=8, dropout=dropout),
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, n_classes)
        )

    def forward(self, x, lengths=None):
        x = self.proj(x)
        x = x.transpose(1, 2)
        x = self.tcn(x)

        if lengths is not None:
            mask = torch.arange(x.shape[2], device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
            mask = mask.unsqueeze(1).float()
            x = (x * mask).sum(dim=2) / mask.sum(dim=2).clamp(min=1)
        else:
            x = x.mean(dim=2)

        return self.classifier(x)

In [30]:
# ============================================================
# CELL 25 — Transformer Model
# ============================================================
class SignTransformer(nn.Module):
    def __init__(self, input_size=708, d_model=256, nhead=4, layers=3, n_classes=25, dropout=0.3):
        super().__init__()

        self.proj = nn.Linear(input_size, d_model)
        self.pos_emb = nn.Embedding(96, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )

        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x, lengths=None):
        B, T, _ = x.shape

        positions = torch.arange(T, device=x.device)
        x = self.proj(x) + self.pos_emb(positions)

        if lengths is not None:
            mask = torch.arange(T, device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)
        else:
            mask = None

        out = self.encoder(x, src_key_padding_mask=mask)

        if lengths is not None:
            valid = (~mask).unsqueeze(-1).float()
            out = (out * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)
        else:
            out = out.mean(dim=1)

        return self.fc(self.norm(out))

In [31]:
# ============================================================
# CELL 26 — CNN + Transformer Model
# ============================================================
class SignCNNTransformer(nn.Module):
    def __init__(self, input_size=708, d_model=256, nhead=4, layers=2, n_classes=25, dropout=0.3):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(input_size, d_model),
            nn.LayerNorm(d_model),
            nn.GELU()
        )

        self.conv = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Conv1d(d_model, d_model, kernel_size=5, padding=2),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.pos_emb = nn.Embedding(96, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )

        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x, lengths=None):
        B, T, _ = x.shape

        x = self.proj(x)

        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)

        positions = torch.arange(T, device=x.device)
        x = x + self.pos_emb(positions)

        if lengths is not None:
            mask = torch.arange(T, device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)
        else:
            mask = None

        out = self.encoder(x, src_key_padding_mask=mask)

        if lengths is not None:
            valid = (~mask).unsqueeze(-1).float()
            out = (out * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)
        else:
            out = out.mean(dim=1)

        return self.fc(self.norm(out))

In [32]:
# ============================================================
# CELL 27 — Model Registry
# ============================================================
model_builders = {
    'MLP': SignMLP,
    'CNN': SignCNN,
    'LSTM': SignLSTM,
    'GRU': SignGRU,
    'CNN_LSTM': SignCNNLSTM,
    'CNN_GRU': SignCNNGRU,
    'BiLSTM_Attention': SignBiLSTMAttention,
    'TCN': SignTCN,
    'Transformer': SignTransformer,
    'CNN_Transformer': SignCNNTransformer,
}

print("Models available:")
for name in model_builders:
    print("-", name)

Models available:
- MLP
- CNN
- LSTM
- GRU
- CNN_LSTM
- CNN_GRU
- BiLSTM_Attention
- TCN
- Transformer
- CNN_Transformer


In [33]:
# ============================================================
# CELL 28 — Dummy Forward Pass Test
# ============================================================
dummy_x = torch.randn(8, 96, 708).to(device)
dummy_l = torch.randint(20, 96, (8,)).to(device)

for name, builder in model_builders.items():
    model = builder(n_classes=N_CLASSES).to(device)

    out = model(dummy_x, dummy_l)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"{name:18s} output={str(out.shape):15s} params={params:,}")

    del model
    torch.cuda.empty_cache()

MLP                output=torch.Size([8, 25]) params=34,939,417
CNN                output=torch.Size([8, 25]) params=390,937
LSTM               output=torch.Size([8, 25]) params=3,569,177
GRU                output=torch.Size([8, 25]) params=2,680,345
CNN_LSTM           output=torch.Size([8, 25]) params=1,774,361
CNN_GRU            output=torch.Size([8, 25]) params=1,511,193
BiLSTM_Attention   output=torch.Size([8, 25]) params=1,992,730
TCN                output=torch.Size([8, 25]) params=1,767,961
Transformer        output=torch.Size([8, 25]) params=1,794,329
CNN_Transformer    output=torch.Size([8, 25]) params=1,793,561


In [34]:
# ============================================================
# CELL 29 — Accuracy Helper
# ============================================================
def topk_acc(logits, targets, k):
    _, pred = logits.topk(k, dim=1)
    correct = pred.eq(targets.view(-1, 1).expand_as(pred))
    return correct.any(dim=1).float().mean().item()

In [35]:
# ============================================================
# CELL 30 — Train One Epoch
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    top1 = 0.0
    top3 = 0.0
    top5 = 0.0

    for x, y, lengths in loader:
        x = x.to(device)
        y = y.to(device)
        lengths = lengths.to(device)

        optimizer.zero_grad()

        logits = model(x, lengths)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        top1 += topk_acc(logits, y, 1)
        top3 += topk_acc(logits, y, min(3, N_CLASSES))
        top5 += topk_acc(logits, y, min(5, N_CLASSES))

    n = len(loader)

    return total_loss / n, top1 / n, top3 / n, top5 / n

In [36]:
# ============================================================
# CELL 31 — Evaluation Function
# ============================================================
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    top1 = 0.0
    top3 = 0.0
    top5 = 0.0

    for x, y, lengths in loader:
        x = x.to(device)
        y = y.to(device)
        lengths = lengths.to(device)

        logits = model(x, lengths)
        loss = criterion(logits, y)

        total_loss += loss.item()
        top1 += topk_acc(logits, y, 1)
        top3 += topk_acc(logits, y, min(3, N_CLASSES))
        top5 += topk_acc(logits, y, min(5, N_CLASSES))

    n = len(loader)

    return total_loss / n, top1 / n, top3 / n, top5 / n

In [37]:
# ============================================================
# CELL 32 — Full Training Function
# ============================================================
def train_model(model, model_name, epochs=30, lr=3e-4):
    model = model.to(device)

    weights_tensor = torch.tensor(class_weights_25, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    best_val_top1 = 0.0
    best_ckpt_path = os.path.join(PROCESSED_DIR, f'best_pilot25_{model_name}.pt')

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        tr_loss, tr_top1, tr_top3, tr_top5 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        vl_loss, vl_top1, vl_top3, vl_top5 = evaluate(
            model, val_loader, criterion, device
        )

        scheduler.step()

        if vl_top1 > best_val_top1:
            best_val_top1 = vl_top1
            torch.save(model.state_dict(), best_ckpt_path)

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"[{model_name}] Epoch {epoch:03d}/{epochs} | "
                f"train_loss={tr_loss:.4f} train_top1={tr_top1:.3f} | "
                f"val_loss={vl_loss:.4f} val_top1={vl_top1:.3f}"
            )

    elapsed_min = (time.time() - start_time) / 60

    model.load_state_dict(torch.load(best_ckpt_path, map_location=device))

    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    te_loss, te_top1, te_top3, te_top5 = evaluate(
        model, test_loader, criterion, device
    )

    return {
        'model': model_name,
        'best_val_top1': best_val_top1,
        'test_loss': te_loss,
        'test_top1': te_top1,
        'test_top3': te_top3,
        'test_top5': te_top5,
        'train_time_min': elapsed_min,
        'params': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'ckpt_path': best_ckpt_path,
    }

In [38]:
# ============================================================
# CELL 33 — Choose Models to Train
# ============================================================
MODELS_TO_TRAIN = [
    'MLP',
    'CNN',
    'LSTM',
    'GRU',
    'CNN_LSTM',
    'CNN_GRU',
    'BiLSTM_Attention',
    'TCN',
    'Transformer',
    'CNN_Transformer',
]

EPOCHS = 30

print("Training these models:")
for m in MODELS_TO_TRAIN:
    print("-", m)

Training these models:
- MLP
- CNN
- LSTM
- GRU
- CNN_LSTM
- CNN_GRU
- BiLSTM_Attention
- TCN
- Transformer
- CNN_Transformer


In [39]:
# ============================================================
# CELL 34 — Train All Selected Models
# ============================================================
results = {}

for model_name in MODELS_TO_TRAIN:
    print("\n" + "=" * 70)
    print(f"Training {model_name}")
    print("=" * 70)

    builder = model_builders[model_name]
    model = builder(n_classes=N_CLASSES)

    results[model_name] = train_model(
        model=model,
        model_name=model_name,
        epochs=EPOCHS,
        lr=3e-4
    )

    print("Done:")
    print(results[model_name])

    del model
    torch.cuda.empty_cache()


Training MLP
[MLP] Epoch 001/30 | train_loss=3.2185 train_top1=0.064 | val_loss=3.1657 val_top1=0.076
[MLP] Epoch 005/30 | train_loss=2.2158 train_top1=0.366 | val_loss=2.4460 val_top1=0.299
[MLP] Epoch 010/30 | train_loss=1.5529 train_top1=0.558 | val_loss=2.2222 val_top1=0.380
[MLP] Epoch 015/30 | train_loss=1.1731 train_top1=0.669 | val_loss=2.1058 val_top1=0.439
[MLP] Epoch 020/30 | train_loss=0.8831 train_top1=0.764 | val_loss=1.9848 val_top1=0.494
[MLP] Epoch 025/30 | train_loss=0.7122 train_top1=0.813 | val_loss=1.9860 val_top1=0.504
[MLP] Epoch 030/30 | train_loss=0.6801 train_top1=0.827 | val_loss=1.9704 val_top1=0.502
Done:
{'model': 'MLP', 'best_val_top1': 0.5167674700419108, 'test_loss': 1.5963499546051025, 'test_top1': 0.5405512703092474, 'test_top3': 0.768715971394589, 'test_top5': 0.8596869330657156, 'train_time_min': 1.4199603994687398, 'params': 34939417, 'ckpt_path': '/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/best

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


[Transformer] Epoch 001/30 | train_loss=2.7861 train_top1=0.187 | val_loss=2.4054 val_top1=0.377
[Transformer] Epoch 005/30 | train_loss=1.2290 train_top1=0.651 | val_loss=1.6903 val_top1=0.576
[Transformer] Epoch 010/30 | train_loss=0.7238 train_top1=0.786 | val_loss=1.5056 val_top1=0.660
[Transformer] Epoch 015/30 | train_loss=0.4295 train_top1=0.876 | val_loss=1.4798 val_top1=0.665
[Transformer] Epoch 020/30 | train_loss=0.2568 train_top1=0.930 | val_loss=1.5242 val_top1=0.684
[Transformer] Epoch 025/30 | train_loss=0.1533 train_top1=0.965 | val_loss=1.5333 val_top1=0.696
[Transformer] Epoch 030/30 | train_loss=0.1221 train_top1=0.970 | val_loss=1.5290 val_top1=0.700
Done:
{'model': 'Transformer', 'best_val_top1': 0.7077956994374593, 'test_loss': 0.7896040285888472, 'test_top1': 0.8026032196848016, 'test_top3': 0.9241719590990167, 'test_top5': 0.9537772222569114, 'train_time_min': 1.893329107761383, 'params': 1794329, 'ckpt_path': '/content/drive/MyDrive/UChicago/Masters/Spring/ML2/

In [40]:
# ============================================================
# CELL 35 — Ablation Table
# ============================================================
rows = []

for name, r in results.items():
    rows.append({
        'Model': r['model'],
        'Best Val Top-1 (%)': round(r['best_val_top1'] * 100, 2),
        'Test Top-1 (%)': round(r['test_top1'] * 100, 2),
        'Test Top-3 (%)': round(r['test_top3'] * 100, 2),
        'Test Top-5 (%)': round(r['test_top5'] * 100, 2),
        'Test Loss': round(r['test_loss'], 4),
        'Train Time (min)': round(r['train_time_min'], 2),
        'Params': r['params'],
        'Checkpoint': r['ckpt_path'],
    })

ablation_df = pd.DataFrame(rows)
ablation_df = ablation_df.sort_values('Test Top-1 (%)', ascending=False)

display(ablation_df)

ablation_path = os.path.join(PROCESSED_DIR, 'ablation_pilot25_all_models.csv')
ablation_df.to_csv(ablation_path, index=False)

print("Saved:", ablation_path)

,Model,Best Val Top-1 (%),Test Top-1 (%),Test Top-3 (%),Test Top-5 (%),Test Loss,Train Time (min),Params,Checkpoint
1,CNN,74.84,83.14,94.88,97.45,0.5854,1.27,390937,/content/drive/MyDrive/UChicago/Masters/Spring...
7,TCN,74.11,81.51,94.31,97.27,0.7288,1.71,1767961,/content/drive/MyDrive/UChicago/Masters/Spring...
5,CNN_GRU,73.79,81.34,93.96,96.61,0.8002,1.40,1511193,/content/drive/MyDrive/UChicago/Masters/Spring...
8,Transformer,70.78,80.26,92.42,95.38,0.7896,1.89,1794329,/content/drive/MyDrive/UChicago/Masters/Spring...
6,BiLSTM_Attention,72.04,79.55,93.08,96.13,0.7968,1.62,1992730,/content/drive/MyDrive/UChicago/Masters/Spring...
3,GRU,69.00,78.89,91.46,95.03,0.8354,2.09,2680345,/content/drive/MyDrive/UChicago/Masters/Spring...
9,CNN_Transformer,74.31,78.80,92.14,95.46,0.8081,1.77,1793561,/content/drive/MyDrive/UChicago/Masters/Spring...
4,CNN_LSTM,72.87,78.24,92.40,94.88,0.8952,1.58,1774361,/content/drive/MyDrive/UChicago/Masters/Spring...
2,LSTM,66.81,75.11,90.44,94.88,1.0278,2.75,3569177,/content/drive/MyDrive/UChicago/Masters/Spring...
0,MLP,51.68,54.06,76.87,85.97,1.5963,1.42,34939417,/content/drive/MyDrive/UChicago/Masters/Spring...


Saved: /content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/ablation_pilot25_all_models.csv


In [41]:
# ============================================================
# CELL 36 — Prediction Helper
# ============================================================
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    for x, y, lengths in loader:
        x = x.to(device)
        lengths = lengths.to(device)

        logits = model(x, lengths)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

    return np.array(all_labels), np.array(all_preds)

In [42]:
# ============================================================
# CELL 37 — Classification Report for Best Model
# ============================================================
best_result = max(results.values(), key=lambda r: r['test_top1'])
best_model_name = best_result['model']

print("Best model:", best_model_name)
print("Test Top-1:", best_result['test_top1'])

best_model = model_builders[best_model_name](n_classes=N_CLASSES).to(device)
best_model.load_state_dict(torch.load(best_result['ckpt_path'], map_location=device))

y_true, y_pred = get_predictions(best_model, test_loader, device)

print(classification_report(
    y_true,
    y_pred,
    target_names=PILOT_SIGNS,
    digits=3
))

Best model: CNN
Test Top-1: 0.8313861144216437
              precision    recall  f1-score   support

     outside      0.771     0.750     0.761        36
        book      0.875     0.875     0.875        40
     another      0.882     0.698     0.779        43
         sad      0.919     1.000     0.958        57
       empty      0.788     0.634     0.703        41
      drawer      0.746     0.898     0.815        49
       dirty      0.878     0.837     0.857        43
         can      0.653     0.800     0.719        40
        room      0.810     0.756     0.782        45
        blue      0.732     0.837     0.781        49
      please      0.945     0.912     0.929        57
    yourself      0.977     0.840     0.903        50
       uncle      1.000     0.930     0.964        57
        make      0.721     0.830     0.772        53
      better      0.809     0.884     0.844        43
      napkin      0.810     0.940     0.870        50
        have      0.782     0.956 

In [43]:
# ============================================================
# CELL 38 — Classification Reports for All Models
# ============================================================
for model_name, r in results.items():
    print("\n" + "=" * 70)
    print(f"Classification Report: {model_name}")
    print("=" * 70)

    model = model_builders[model_name](n_classes=N_CLASSES).to(device)
    model.load_state_dict(torch.load(r['ckpt_path'], map_location=device))

    y_true, y_pred = get_predictions(model, test_loader, device)

    print(classification_report(
        y_true,
        y_pred,
        target_names=PILOT_SIGNS,
        digits=3
    ))

    del model
    torch.cuda.empty_cache()


Classification Report: MLP
              precision    recall  f1-score   support

     outside      0.455     0.556     0.500        36
        book      0.459     0.425     0.442        40
     another      0.372     0.372     0.372        43
         sad      0.764     0.737     0.750        57
       empty      0.526     0.244     0.333        41
      drawer      0.554     0.633     0.590        49
       dirty      0.674     0.721     0.697        43
         can      0.524     0.550     0.537        40
        room      0.233     0.156     0.187        45
        blue      0.289     0.265     0.277        49
      please      0.786     0.579     0.667        57
    yourself      0.414     0.480     0.444        50
       uncle      0.754     0.860     0.803        57
        make      0.408     0.377     0.392        53
      better      0.300     0.349     0.323        43
      napkin      0.596     0.620     0.608        50
        have      0.532     0.911     0.672        45

In [44]:
# ============================================================
# CELL 39 — Export Best Model to ONNX
# ============================================================
!pip install onnx onnxruntime onnxscript -q

import onnxruntime as ort

best_model = model_builders[best_model_name](n_classes=N_CLASSES).to(device)
best_model.load_state_dict(torch.load(best_result['ckpt_path'], map_location=device))
best_model.eval()

dummy_x = torch.randn(1, 96, 708).to(device)
dummy_l = torch.tensor([96]).to(device)

onnx_path = os.path.join(PROCESSED_DIR, f'sign_model_pilot25_{best_model_name}.onnx')

torch.onnx.export(
    best_model,
    (dummy_x, dummy_l),
    onnx_path,
    input_names=['x', 'lengths'],
    output_names=['logits'],
    dynamic_axes={
        'x': {0: 'batch_size'},
        'lengths': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
    },
    opset_version=17
)

print("Saved ONNX:", onnx_path)

sess = ort.InferenceSession(onnx_path)

out = sess.run(None, {
    'x': dummy_x.cpu().numpy(),
    'lengths': dummy_l.cpu().numpy(),
})

print("ONNX output shape:", out[0].shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.1 MB/s eta 0:00:00


/tmp/ipykernel_704/515103483.py:17: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0519 13:14:28.060000 704 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SignCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SignCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:460: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Saved ONNX: /content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/sign_model_pilot25_CNN.onnx
ONNX output shape: (1, 25)


In [45]:
# ============================================================
# CELL 40 — Save Final Training Summary
# ============================================================
pilot_label_map.update({
    'best_model': best_model_name,
    'best_model_checkpoint': best_result['ckpt_path'],
    'onnx_path': onnx_path,
    'test_top1': float(best_result['test_top1']),
    'test_top3': float(best_result['test_top3']),
    'test_top5': float(best_result['test_top5']),
})

with open(label_map_path, 'w') as f:
    json.dump(pilot_label_map, f, indent=2)

summary_lines = [
    "=" * 60,
    "SIGNBRIDGE PILOT-25 TRAINING SUMMARY",
    "=" * 60,
    "",
    f"Number of signs: {N_CLASSES}",
    f"Signs: {', '.join(PILOT_SIGNS)}",
    "",
    f"Train samples: {len(X_tr):,}",
    f"Val samples: {len(X_vl):,}",
    f"Test samples: {len(X_te):,}",
    "",
    f"Best model: {best_model_name}",
    f"Best model test top-1: {best_result['test_top1'] * 100:.2f}%",
    f"Best model test top-3: {best_result['test_top3'] * 100:.2f}%",
    f"Best model test top-5: {best_result['test_top5'] * 100:.2f}%",
    "",
    "Files saved:",
    "  pilot25_label_map.json",
    "  class_weights_25.npy",
    "  ablation_pilot25_all_models.csv",
    f"  sign_model_pilot25_{best_model_name}.onnx",
    "",
    "=" * 60,
]

summary_text = "\n".join(summary_lines)

print(summary_text)

summary_path = os.path.join(PROCESSED_DIR, 'pilot25_training_summary.txt')

with open(summary_path, 'w') as f:
    f.write(summary_text)

print("Saved:", summary_path)

SIGNBRIDGE PILOT-25 TRAINING SUMMARY

Number of signs: 25
Signs: outside, book, another, sad, empty, drawer, dirty, can, room, blue, please, yourself, uncle, make, better, napkin, have, apple, any, bird, dad, doll, kitty, noisy, yucky

Train samples: 5,611
Val samples: 927
Test samples: 1,181

Best model: CNN
Best model test top-1: 83.14%
Best model test top-3: 94.88%
Best model test top-5: 97.45%

Files saved:
  pilot25_label_map.json
  class_weights_25.npy
  ablation_pilot25_all_models.csv
  sign_model_pilot25_CNN.onnx

Saved: /content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/pilot25_training_summary.txt


| Model            | Test Accuracy |
| ---------------- | ------------: |
| CNN              |     **83.1%** |
| CNN_GRU          |         81.7% |
| TCN              |         81.4% |
| Transformer      |         80.2% |
| BiLSTM_Attention |         79.8% |
| GRU              |         79.1% |
| CNN_Transformer  |         78.6% |
| CNN_LSTM         |         78.4% |
| LSTM             |         75.2% |
| MLP              |         54.0% |



Based on this I will prioritize these four:
| Priority | Model           | Why                                                                     |
| -------- | --------------- | ----------------------------------------------------------------------- |
| 1        | **CNN**         | Best overall accuracy (83.1%); fastest and simplest for deployment      |
| 2        | **CNN_GRU**     | Strong hybrid model (81.7%); captures temporal dependencies well        |
| 3        | **TCN**         | High performance (81.4%); efficient sequence modeling with convolutions |
| 4        | **Transformer** | Strong attention-based architecture for global temporal relationships   |
